# Doc Compass: which doctor do I book?

Describe a health concern in your own words and get the kind of doctor to book. The app checks
for emergency signs first, then routes the concern, then explains the result. It suggests a
door to knock on; it does not diagnose.

Run the cells in order. The last one prints a public link and its QR code. The link works
while this notebook is running.

### 1. Install packages

**Use a GPU runtime** (Runtime → Change runtime type → T4 GPU) so the language model can write
the explanations. On a CPU runtime the app still works and uses its built-in explanation text.

In [ ]:
%pip install -q "gradio==6.28.0" "transformers==5.18.0" "scikit-learn==1.9.1" "joblib==1.6.0" huggingface_hub segno

### 2. Download the app

The code and the trained routers come from one pinned Hugging Face repo.

In [ ]:
import sys
from huggingface_hub import snapshot_download

BUNDLE = {"repo_id": "akshara-ns/doc-compass", "revision": "3c47409d872420e09fc3ef37108158bfcdd23309"}  # pinned commit
bundle_dir = snapshot_download(**BUNDLE)

# Forget any copy loaded earlier in this session, so re-running this cell picks up this version.
for name in [name for name in sys.modules if name.split(".")[0] == "doccompass"]:
    del sys.modules[name]
sys.path = [bundle_dir] + [path for path in sys.path if "--doc-compass" not in path]
print("App downloaded to", bundle_dir)

### 3. Load the models

- **Router:** BiomedBERT, fine-tuned by us on our labelled posts. If it can't load, the app uses our
  from-scratch router (TF-IDF + logistic regression) instead.
- **Explanation:** Qwen2.5-1.5B-Instruct, used as downloaded. Its text is checked before it is
  shown, and the app falls back to fixed wording when a check fails.

In [ ]:
import torch
from doccompass.app import EXAMPLES, launch
from doccompass.explain import QwenExplainer
from doccompass.pipeline import load_router, route_concern

router = load_router()
assert router is not None, "No router loaded: check that the download in step 2 finished."
print("Router:", type(router).__name__)

explainer = None
if torch.cuda.is_available():
    try:
        explainer = QwenExplainer()
    except Exception as error:  # the app still works without it
        print("Language model unavailable:", error)
print("Explanation:", explainer.name if explainer else "built-in wording (no GPU)")

### 4. Check it works

Three made-up concerns: an ordinary one, a vague one and an emergency.

In [ ]:
checks = [EXAMPLES[0], "I feel tired all the time and I don't know why.", EXAMPLES[-1]]
for text in checks:
    result = route_concern(text, router, explainer)
    top = ", ".join(f"{option['label']} {option['confidence']:.0%}" for option in result["options"])
    print(f"{result['status']:>9} | {top or result['message']} | {text[:60]}")

assert route_concern(checks[0], router)["status"] == "routed"
assert route_concern(checks[-1], router)["status"] == "emergency"

### 5. Launch

Opens the app below and prints a temporary public link with its QR code.

In [ ]:
import segno
from IPython.display import Image, display

_, _, public_url = launch(router, explainer, share=True, prevent_thread_lock=True)
print("Public link:", public_url)

segno.make(public_url).save("doc_compass_qr.png", scale=10, border=2)
display(Image("doc_compass_qr.png"))

### Notes

- Nothing typed into the app is stored.
- The emergency rules quote published warning signs (MedlinePlus and the CDC) and run before
  any model.
- The router was trained on a public set of short patient comments, so it can be confidently
  wrong on wording that set doesn't cover.
- The public link and its QR code are new each time this notebook runs.
- To stop the app, run `import gradio; gradio.close_all()`.